# Интерпретируемость: Permutation Feature Importance

Этот ноутбук отвечает на вопрос: **«На какие индикаторы смотрит нейросеть при принятии решений?»**

Мы берем уже обученную модель и прогоняем её на тестовых данных. Затем мы по очереди берем каждую колонку (например, `RSI`), перемешиваем её (превращая в мусор) и снова прогоняем модель. Чем сильнее падает прибыль без этой колонки — тем важнее она для агента.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack

from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5

In [2]:
from core.experiment_manager import get_eval_widgets, get_experiment_paths
exp_widget = get_eval_widgets()

# Автоматическое получение путей
model_path, norm_path, tensorboard_log = get_experiment_paths(exp_widget.value)

# Поддержка старых переменных в ноутбуке
model_widget = type('obj', (object,), {'value': model_path})
norm_widget = type('obj', (object,), {'value': norm_path})
MODEL_PATH = model_path
NORM_PATH = norm_path
vec_normalize_path = norm_path


Dropdown(description='Experiment:', layout=Layout(width='50%'), options=('ppo_prod', 'ppo_prod_cnn_2m', 'ppo_g…

## 1. Загрузка Данных и Модели

In [3]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2022-01-01', interval='4h', source='bybit_futures')
fg = FeatureGenerator()
data_features = fg.transform(df)

# Откусываем тестовый кусок
TEST_SIZE = 360
test_df = data_features.iloc[-TEST_SIZE:].reset_index(drop=True)

# ==========================================
# НАСТРОЙКИ МОДЕЛИ (ВАЖНО!)
# ==========================================
USE_FRAME_STACK = True
N_STACK = 10 

try:
    # ПОМЕНЯЙТЕ ПУТЬ НА СВОЙ!
    model = PPO.load(model_widget.value) 
    vec_normalize_path = norm_widget.value 
    print("Модель успешно загружена!")
except Exception as e:
    print(f"Ошибка загрузки: {e}")

✅ All data in cache, loading from disk...
✅ HMM Model loaded from /Users/maximsinyaev/projects/trading_rl/trading_rl/models/hmm_model.pkl
Модель успешно загружена!


## 2. Функция для замера PnL

## 3. Расчет Важности Признаков (Permutation)

In [7]:
from core.evaluation.validation import evaluate_model, calculate_permutation_importance, plot_boruta_importance

# 1. Считаем базовый PnL
print("Считаем базовый PnL (без вмешательств)...")
base_pnl = evaluate_model(model, vec_normalize_path, test_df, USE_FRAME_STACK, N_STACK, num_seeds=10)
print(f"Базовый баланс: ${base_pnl:,.2f}")

# 2. Выбираем фичи
base_features = [
    'open_over_ema_20', 'high_over_ema_20', 'low_over_ema_20', 'close_over_ema_20',
    'log_return', 'rsi_norm', 'normalized_volume',
    'macd_line_norm', 'macd_signal_norm', 'macd_hist_norm',
    'gk_volatility', 'frac_diff_norm',
    'funding_rate', 'funding_delta', 'oi_delta'
]
ema_features = ['ema_50_norm', 'ema_100_norm']
ema_diff_features = ['ema_diff_20_50', 'ema_diff_20_100', 'ema_diff_50_100']
hmm_features = [c for c in test_df.columns if 'hmm_regime' in c]

final_features = [f for f in base_features + ema_features + ema_diff_features + hmm_features if f in test_df.columns]

# 3. Расчет важности
importance_scores = calculate_permutation_importance(
    model, vec_normalize_path, test_df, final_features, base_pnl, 
    n_iterations=100, use_frame_stack=USE_FRAME_STACK, n_stack=N_STACK, num_seeds=10
)

Считаем базовый PnL (без вмешательств)...
Базовый баланс: $91,007.68
\nНачинаем перемешивание фичей (100 итераций на каждую, 10 seed-прогонов внутри)...\n


Features:   0%|          | 0/23 [00:00<?, ?it/s]

Фича open_over_ema_20: среднее падение = $125.28 ± $130.94
Фича high_over_ema_20: среднее падение = $-6.40 ± $78.47
Фича low_over_ema_20: среднее падение = $273.06 ± $175.79
Фича close_over_ema_20: среднее падение = $146.42 ± $141.92
Фича log_return: среднее падение = $259.06 ± $812.46
Фича rsi_norm: среднее падение = $-40.24 ± $265.97
Фича normalized_volume: среднее падение = $730.76 ± $935.57
Фича macd_line_norm: среднее падение = $631.73 ± $412.42
Фича macd_signal_norm: среднее падение = $877.24 ± $547.50
Фича macd_hist_norm: среднее падение = $3.58 ± $287.27
Фича gk_volatility: среднее падение = $189.97 ± $367.20
Фича frac_diff_norm: среднее падение = $64.94 ± $566.86
Фича funding_rate: среднее падение = $191.20 ± $134.77
Фича funding_delta: среднее падение = $92.95 ± $173.41
Фича oi_delta: среднее падение = $1,132.71 ± $883.65
Фича ema_50_norm: среднее падение = $63.57 ± $168.43
Фича ema_100_norm: среднее падение = $137.95 ± $166.88
Фича ema_diff_20_50: среднее падение = $413.77 ±

## 4. График Важности

In [6]:
from core.evaluation.validation import plot_boruta_importance

# Рисуем крутые ящики с усами (Boruta Boxplots)
plot_boruta_importance(importance_scores)



ИНСТРУКЦИЯ:
1. Весь ЯЩИК ПРАВЕЕ нуля: Фича ПОЛЕЗНАЯ. Удалять нельзя!
2. Ящик ПЕРЕСЕКАЕТ ноль: Фича — ШУМ. Модель не уверена в её пользе.
3. Весь ЯЩИК ЛЕВЕЕ нуля: Фича ВРЕДНАЯ. Удаление этой колонки увеличит прибыль!
